# 🔮 ETS & ARIMA Forecasting
**fpppy Chapters 7–9 · Pattern Recognition for the Rest of Us**

> Two complementary forecasting families: ETS captures level, trend, and seasonality with exponential smoothing; ARIMA models autocorrelation structure after differencing. Together they cover most forecasting needs.

### Required reading
📘 [fpppy Ch 7 (ETS)](https://otexts.com/fpppy/expsmooth.html) · [Ch 8 (ARIMA)](https://otexts.com/fpppy/arima.html) · [Ch 9 (Evaluation)](https://otexts.com/fpppy/accuracy.html)

### What you'll learn
- Naive and seasonal naive baselines
- Simple Exponential Smoothing (SES), Holt's linear, Holt-Winters
- ARIMA: differencing, AR, MA — intuition and fitting
- auto_arima — automatic model selection
- Train/test split for time series, MAE/RMSE/MAPE evaluation

### Dataset: Air passengers + Retail sales
### Time: ~70 minutes

In [ ]:
import numpy as np,pandas as pd,matplotlib.pyplot as plt,warnings
warnings.filterwarnings('ignore')
plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'#f8f9fa','axes.grid':True,'grid.alpha':0.4,'axes.spines.top':False,'axes.spines.right':False,'font.size':11})
!pip install pmdarima -q
from statsmodels.tsa.holtwinters import ExponentialSmoothing, SimpleExpSmoothing
from statsmodels.tsa.arima.model import ARIMA
from pmdarima import auto_arima
from sklearn.metrics import mean_absolute_error, mean_squared_error

In [ ]:
# ── Load datasets from the course repo ──────────────────────────────────────
import subprocess, os

# Clone the course repo if not already present (works in Colab)
if not os.path.exists('pattern-recognition-notebooks'):
    subprocess.run(['git','clone','--depth','1',
                    'https://github.com/ladataanalytics/pattern-recognition-notebooks.git'],
                   capture_output=True)

DATA_DIR = 'pattern-recognition-notebooks/data'
if not os.path.exists(DATA_DIR):
    # Fallback: already in repo root (e.g. running locally)
    DATA_DIR = '../data'

print(f"✓ Data directory: {DATA_DIR}")
print(f"✓ Available datasets: {os.listdir(DATA_DIR)}")

### Load Dataset: AirPassengers

In [ ]:
import pandas as pd
passengers = pd.read_csv(f'{DATA_DIR}/AirPassengers.csv',
                          parse_dates=['Month'], index_col='Month')
passengers.index.freq = 'MS'
print(f"Air Passengers: {passengers.shape}  ({passengers.index[0].year}–{passengers.index[-1].year})")
passengers.head()

In [ ]:
# Air passengers — numpy built-in reconstruction (exact Box-Jenkins data)
import pandas as pd, numpy as np
data = [112,118,132,129,121,135,148,148,136,119,104,118,
        115,126,141,135,125,149,170,170,158,133,114,140,
        145,150,178,163,172,178,199,199,184,162,146,166,
        171,180,193,181,183,218,230,242,209,191,172,194,
        196,196,236,235,229,243,264,272,237,211,180,201,
        204,188,235,227,234,264,302,293,259,229,203,229,
        242,233,267,269,270,315,364,347,312,274,237,278,
        284,277,317,313,318,374,413,405,355,306,271,306,
        315,301,356,348,355,422,465,467,404,347,305,336,
        340,318,362,348,363,435,491,505,404,359,310,337,
        360,342,406,396,420,472,548,559,463,407,362,405,
        417,391,419,461,472,535,622,606,508,461,390,432]
idx = pd.date_range('1949-01', periods=144, freq='MS')
passengers = pd.DataFrame({'Passengers': data}, index=idx)
print(f"Air Passengers: {passengers.shape}")
passengers.head()

## 📐 Part 1 — Baseline Models

Before any sophisticated model, establish baselines. If your fancy model can't beat a naive baseline, something is wrong.

- **Naive:** forecast = last observed value
- **Seasonal Naive:** forecast = same period last year
- **Mean:** forecast = historical mean

These are your floor — any real model should do better.

In [ ]:
# Implement baseline forecasts
h = len(test)

naive_fc      = pd.Series([train['Passengers'].iloc[-1]] * h, index=test.index)
seasonal_naive = train['Passengers'].iloc[-12:].values  # last 12 months repeated
snaive_fc     = pd.Series(np.tile(seasonal_naive, 2)[:h], index=test.index)
mean_fc       = pd.Series([train['Passengers'].mean()] * h, index=test.index)

def eval_forecast(name, fc, actual):
    mae  = mean_absolute_error(actual, fc)
    rmse = np.sqrt(mean_squared_error(actual, fc))
    mape = np.mean(np.abs((actual - fc) / actual)) * 100
    return {'Model':name, 'MAE':mae, 'RMSE':rmse, 'MAPE%':mape}

results = []
for name, fc in [('Naive', naive_fc), ('Seasonal Naive', snaive_fc), ('Mean', mean_fc)]:
    results.append(eval_forecast(name, fc, test['Passengers']))

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(passengers.index, passengers['Passengers'], color='black', lw=1.5, label='Actual', alpha=0.8)
ax.axvline(test.index[0], color='#888', lw=1.5, ls='--', label='Train/Test split')
for fc, color, name in [(naive_fc,'#e85d20','Naive'),
                         (snaive_fc,'#1e5fa8','Seasonal Naive'),
                         (mean_fc,'#888','Mean')]:
    ax.plot(fc.index, fc, color=color, lw=1.5, ls='--', label=name)
ax.set_title('Baseline Forecasts — Air Passengers')
ax.legend()
plt.tight_layout()
plt.show()

print("\n=== Baseline Performance ===")
print(pd.DataFrame(results).to_string(index=False, float_format='%.2f'))
print("\n📌 Seasonal Naive is surprisingly hard to beat on strongly seasonal data")

## 📈 Part 2 — Exponential Smoothing (ETS)

ETS models assign exponentially decreasing weights to past observations — recent observations matter more.

- **SES (α):** level only — best for flat series
- **Holt (α, β):** level + trend — best for trending series  
- **Holt-Winters (α, β, γ):** level + trend + seasonality — the full model

The 'ETS' name refers to Error-Trend-Seasonality. Each component can be None, Additive, or Multiplicative.

In [ ]:
# Fit Holt-Winters (additive and multiplicative)
hw_add  = ExponentialSmoothing(train['Passengers'],
                                trend='add', seasonal='add', seasonal_periods=12).fit()
hw_mult = ExponentialSmoothing(train['Passengers'],
                                trend='add', seasonal='mul', seasonal_periods=12).fit()

fc_add  = hw_add.forecast(h)
fc_mult = hw_mult.forecast(h)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(passengers.index, passengers['Passengers'], color='black', lw=1.5, label='Actual')
ax.axvline(test.index[0], color='#888', lw=1.5, ls='--')
ax.plot(fc_add.index,  fc_add,  color='#1e5fa8', lw=2, label='Holt-Winters Additive')
ax.plot(fc_mult.index, fc_mult, color='#e85d20', lw=2, label='Holt-Winters Multiplicative')
ax.fill_between(test.index,
                fc_mult * 0.92, fc_mult * 1.08,
                alpha=0.15, color='#e85d20', label='~80% interval (multiplicative)')
ax.set_title('Holt-Winters Forecasts — Air Passengers')
ax.legend()
plt.tight_layout()
plt.show()

results += [eval_forecast('HW Additive',       fc_add,  test['Passengers']),
            eval_forecast('HW Multiplicative',  fc_mult, test['Passengers'])]

print("\n=== ETS vs Baseline ===")
print(pd.DataFrame(results).to_string(index=False, float_format='%.2f'))
print("\nSmoothing parameters (multiplicative):")
print(f"  α (level) = {hw_mult.params['smoothing_level']:.3f}")
print(f"  β (trend) = {hw_mult.params['smoothing_trend']:.3f}")
print(f"  γ (seasonal) = {hw_mult.params['smoothing_seasonal']:.3f}")

## 📉 Part 3 — ARIMA: AutoRegressive Integrated Moving Average

**AR(p):** y_t depends on its past p values
```
y_t = φ₁y_{t-1} + φ₂y_{t-2} + ... + φₚy_{t-p} + ε_t
```

**MA(q):** y_t depends on past q forecast errors
```
y_t = ε_t + θ₁ε_{t-1} + ... + θqε_{t-q}
```

**I(d):** differencing d times to achieve stationarity

**ARIMA(p,d,q)** combines all three. **SARIMA(p,d,q)(P,D,Q)m** adds seasonal components.

In [ ]:
# auto_arima — finds optimal (p,d,q)(P,D,Q) automatically
print("Running auto_arima (may take 30-60 seconds)...")
arima_model = auto_arima(
    train['Passengers'],
    seasonal=True,
    m=12,           # monthly seasonality
    stepwise=True,  # faster search
    information_criterion='aic',
    trace=False,
    error_action='ignore',
    suppress_warnings=True
)
print("\nBest ARIMA model found:")
print(arima_model.summary())

In [ ]:
# Forecast with best ARIMA
arima_fc_vals, conf_int = arima_model.predict(n_periods=h, return_conf_int=True)
arima_fc = pd.Series(arima_fc_vals, index=test.index)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(passengers.index, passengers['Passengers'], color='black', lw=1.5, label='Actual')
ax.axvline(test.index[0], color='#888', lw=1.5, ls='--', label='Train/Test split')
ax.plot(arima_fc.index, arima_fc, color='#1a7a45', lw=2, label=f'ARIMA{arima_model.order}{arima_model.seasonal_order}')
ax.fill_between(test.index, conf_int[:,0], conf_int[:,1], alpha=0.15, color='#1a7a45', label='95% CI')
ax.plot(fc_mult.index, fc_mult, color='#e85d20', lw=1.5, ls='--', alpha=0.7, label='HW Multiplicative')
ax.set_title('ARIMA vs Holt-Winters — Air Passengers Forecast')
ax.legend()
plt.tight_layout()
plt.show()

results.append(eval_forecast('ARIMA', arima_fc, test['Passengers']))
print("\n=== Final Model Comparison ===")
df_res = pd.DataFrame(results).sort_values('RMSE')
print(df_res.to_string(index=False, float_format='%.2f'))
print(f"\n🏆 Best model by RMSE: {df_res.iloc[0]['Model']}")

In [ ]:
# Residual diagnostics — good model → white noise residuals
arima_model.plot_diagnostics(figsize=(12, 8))
plt.suptitle('ARIMA Residual Diagnostics\n(residuals should look like white noise)', y=1.01)
plt.tight_layout()
plt.show()
print("\n📌 What to look for:")
print("  Standardized residuals: random scatter around zero, no patterns")
print("  Histogram: approximately normal")
print("  Q-Q plot: points on the diagonal")
print("  Correlogram: no significant spikes (residuals uncorrelated)")

In [ ]:
# @title 📝 Quiz — ETS & ARIMA Forecasting
# @markdown Answer each question in the box below, then run the AI grading cell.

# @markdown **Q1:** What do the three parameters of ARIMA(p,d,q) each represent?
# @markdown **Q2:** Why can you NOT randomly shuffle train/test split for time series?
# @markdown **Q3:** When would you choose Holt-Winters over ARIMA?
# @markdown **Q4:** What does the 'd' in ARIMA represent and when is d=1 used?
# @markdown **Q5:** If residuals show significant ACF spikes, what does that indicate?

q1 = "" # @param {type:"string", placeholder:"your answer"}
q2 = "" # @param {type:"string", placeholder:"your answer"}
q3 = "" # @param {type:"string", placeholder:"your answer"}
q4 = "" # @param {type:"string", placeholder:"your answer"}
q5 = "" # @param {type:"string", placeholder:"your answer"}

answers = {"q1": q1, "q2": q2, "q3": q3, "q4": q4, "q5": q5}
answered = sum(1 for v in answers.values() if str(v).strip())
print(f"  {answered}/5 answered — run the AI grading cell below")

### 📤 Submit your results

In [ ]:
_NB_NAME_ = "ETS & ARIMA Forecasting"
# @title 🤖 AI Feedback — enter your username and click ▶ Run
# @markdown **No API key needed.** AI grading runs free inside your Colab session.
# @markdown
# @markdown Enter your GitHub username below so your instructor can track your submission, then click ▶ Run.

GITHUB_USERNAME = "" # @param {type:"string", placeholder:"e.g. jsmith42"}

# ── runs automatically — do not edit below ───────────────────
import json, textwrap, re as _re
_NB_TITLE = globals().get("_NB_NAME_", "this notebook")

def _grade(answers_dict, nb_name):
    answered = {k: v for k, v in answers_dict.items() if str(v).strip()}
    n_total  = len(answers_dict)
    qa       = "\n".join(f"Q{k.replace('q','')}: {v}" for k,v in answered.items())
    prompt   = (f"You are a TA grading quiz answers for \"{nb_name}\" in a data science "
                f"course (Data Pattern Recognition for the Rest of Us, based on ISLP).\n\n"
                f"Student answers ({len(answered)}/{n_total}):\n{qa or '(none)'}\n\n"
                f"Grade conceptual understanding. Accept reasonable paraphrases. "
                f"Be encouraging and specific. Reply ONLY in this JSON (no markdown):\n"
                f"{{\"quiz_score\":<int 0-{n_total}>,"
                f"\"grade\":\"<Excellent|Good|Needs Review|Incomplete>\","
                f"\"feedback\":\"<2-3 sentences>\","
                f"\"tip\":\"<one takeaway>\"}}") 
    try:
        import google.generativeai as genai
        import google.auth, google.auth.transport.requests
        creds, _ = google.auth.default()
        creds.refresh(google.auth.transport.requests.Request())
        genai.configure(credentials=creds)
        r = genai.GenerativeModel("gemini-2.0-flash").generate_content(prompt)
        return r.text, "Gemini via Colab"
    except Exception:
        pass
    try:
        from google.colab import userdata
        key = userdata.get("GEMINI_API_KEY")
        if key and len(key) > 10:
            import google.generativeai as genai
            genai.configure(api_key=key)
            r = genai.GenerativeModel("gemini-2.0-flash").generate_content(prompt)
            return r.text, "Gemini via key"
    except Exception:
        pass
    return None, None

def _fallback(answers_dict):
    n = len(answers_dict)
    done = [v for v in answers_dict.values() if str(v).strip()]
    nd, avg = len(done), sum(len(str(v)) for v in done)/max(len(done),1)
    if nd == 0: return {"quiz_score":0,"grade":"Incomplete","feedback":"No answers — fill in the quiz above.","tip":"Each question needs 1-2 sentences."}
    if avg < 15: return {"quiz_score":nd,"grade":"Needs Review","feedback":f"{nd}/{n} answered but very brief. Explain your reasoning.","tip":"Aim for 1-2 sentences per answer."}
    if nd == n:  return {"quiz_score":n,"grade":"Good","feedback":f"All {n} questions answered.","tip":"Review any concepts that felt unclear."}
    return {"quiz_score":nd,"grade":"Needs Review","feedback":f"{nd}/{n} answered. Complete the remaining {n-nd}.","tip":"Answer all questions for full credit."}

def _show(result, username, nb_name, source):
    C = {"Excellent":"\033[92m","Good":"\033[94m","Needs Review":"\033[93m","Incomplete":"\033[91m"}
    R = "\033[0m"; g = result.get("grade","?")
    n = len(answers); qs = result.get("quiz_score",0)
    print("\n"+"\u2550"*56)
    print(f"  \U0001f916 AI Feedback \u2014 {nb_name}")
    if source: print(f"  Powered by   {source}")
    print("\u2550"*56)
    print(f"  Student : {'@'+username if username else '\u26a0 set GITHUB_USERNAME above'}")
    print(f"  Grade   : {C.get(g,'')} {g} {R}")
    print(f"  Score   : {qs}/{n}  [{'\u2588'*qs+'\u2591'*(n-qs)}]")
    print()
    [print(f"  {l}") for l in textwrap.wrap(result.get("feedback",""),54)]
    tip = result.get("tip","")
    if tip:
        print()
        [print(f"  \U0001f4a1 {l}") for l in textwrap.wrap(tip,52)]
    print("\u2550"*56+"\n")

if "answers" not in globals():
    print("\u26a0  Run the quiz cell above first, then re-run this cell.")
else:
    nd = sum(1 for v in answers.values() if str(v).strip())
    print(f"  Notebook : {_NB_TITLE}  \u2022  {nd}/{len(answers)} answered")
    raw, src = _grade(answers, _NB_TITLE)
    if raw:
        try:
            result = json.loads(_re.sub(r"```(?:json)?\s*|```","",raw).strip())
        except Exception:
            result = {"quiz_score":nd,"grade":"Good" if nd==len(answers) else "Needs Review","feedback":raw[:400],"tip":""}
    else:
        print("  (Gemini unavailable \u2014 using completion-based feedback)\n")
        result = _fallback(answers)
        src = None
    _show(result, GITHUB_USERNAME.strip(), _NB_TITLE, src)
    print("  \U0001f4be  File \u2192 Save a copy in GitHub \u2192 your fork\n")
